In [1]:
# Setup
import sys
import warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from recsys.config import load_config, set_seed
from recsys import plots
cfg = load_config()
set_seed(cfg["seed"])
plots.set_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
import json
import os
from fastapi.testclient import TestClient
from recsys.serving import build_serving
from recsys import genai as G

In [2]:
# 1. Batch-score the test-week customers and write the API artifacts
out = build_serving(cfg)
json.loads((out / "manifest.json").read_text())

{'model': 'lgbm_classifier',
 'as_of': '2020-09-15',
 'customers': 14023,
 'k': 12}

In [3]:
# 2. One demo customer: top 12 with template reasons
recs = pd.read_parquet(out / "recs.parquet")
arts = pd.read_parquet(out / "articles.parquet").set_index("article_id")
demo = json.loads((out / "demo_customers.json").read_text())[0]
one = recs[recs["customer_id"] == demo].join(arts[["prod_name", "product_type_name", "colour_group_name"]], on="article_id")
one[["rank", "prod_name", "product_type_name", "colour_group_name", "reason", "bought_next_week"]]

,rank,prod_name,product_type_name,colour_group_name,reason,bought_next_week
167868,1,Jade HW Skinny Denim TRS,Trousers,Dark Blue,Buy it again: you have ordered this before,True
167869,2,Jade HW Skinny Denim TRS,Trousers,Dark Blue,Buy it again: you have ordered this before,True
167870,3,Jade HW Skinny Denim TRS,Trousers,Black,A new color of something you already love,False
167871,4,Jade HW Skinny Denim TRS,Trousers,Black,A new color of something you already love,False
167872,5,Jade HW Skinny Denim TRS,Trousers,Light Blue,A new color of something you already love,False
167873,6,Jade HW Skinny Denim TRS,Trousers,Dark Blue,A new color of something you already love,False
167874,7,LW (J) Mahalia Consc S-shirt,Bra,Dark Green,Buy it again: you have ordered this before,False
167875,8,Pineapple sweater,Sweater,Light Turquoise,Buy it again: you have ordered this before,False
167876,9,Ilse padded softbra,Bra,Black,Buy it again: you have ordered this before,False
167877,10,Frida push up seamless,Unknown,Black,Buy it again: you have ordered this before,False


In [4]:
# 3. How often each reason type is used across all customers
recs["reason"].value_counts(normalize=True)

reason
Trending now: sales are climbing this week   0.2139
Popular with shoppers right now              0.1919
Buy it again: you have ordered this before   0.1442
A new color of something you already love    0.1423
One of this week's best sellers              0.1317
In the colors you usually go for             0.0944
Matches the kind of pieces you buy most      0.0559
Picked to match your style                   0.0161
Often bought together with your past picks   0.0097
Name: proportion, dtype: float64

In [5]:
# 4. LLM reasons (uses Claude when ANTHROPIC_API_KEY is set, otherwise the template)
sample_rows = one.head(4)
pd.DataFrame([dict(zip(["reason", "source"], G.llm_reason(f"{r.prod_name} ({r.product_type_name}, {r.colour_group_name})", r.facts.split("; ")))) | {"item": r.prod_name, "facts": r.facts} for r in sample_rows.itertuples()])

,reason,source,item,facts
0,Buy it again: you have ordered this before,template,Jade HW Skinny Denim TRS,the shopper bought this exact item before
1,Buy it again: you have ordered this before,template,Jade HW Skinny Denim TRS,the shopper bought this exact item before
2,A new color of something you already love,template,Jade HW Skinny Denim TRS,it is another color or version of a trousers t...
3,A new color of something you already love,template,Jade HW Skinny Denim TRS,it is another color or version of a trousers t...


In [6]:
# 5. Guardrail: generated lines that mention personal traits are replaced by the template
pd.DataFrame({"text": ["Loved by women your age", "Great for young shoppers like you", "A new color of something you already love", "Often bought together with your past picks"]}).assign(passes=lambda d: d["text"].map(G.is_safe))

,text,passes
0,Loved by women your age,False
1,Great for young shoppers like you,False
2,A new color of something you already love,True
3,Often bought together with your past picks,True


In [7]:
# 6. LLM-assisted data dictionary: the column profile we send to the model
print(G.column_profile(pd.read_parquet(cfg["paths"]["processed"] / "customers_sample.parquet").drop(columns=["customer_id", "postal_code"]))[:1500])

FN: {"dtype": "float64", "missing_pct": 65.18, "unique": 1, "min": 1.0, "max": 1.0}
Active: {"dtype": "float64", "missing_pct": 66.12, "unique": 1, "min": 1.0, "max": 1.0}
club_member_status: {"dtype": "str", "missing_pct": 0.44, "unique": 3, "examples": ["ACTIVE", "PRE-CREATE", "LEFT CLUB"]}
fashion_news_frequency: {"dtype": "str", "missing_pct": 1.17, "unique": 4, "examples": ["NONE", "Regularly", "Monthly", "None"]}
age: {"dtype": "float64", "missing_pct": 1.16, "unique": 83, "min": 16.0, "max": 99.0}


In [8]:
# 7. API smoke test
os.environ["RECSYS_SERVING_DIR"] = str(out)
from recsys import api
api.store.cache_clear()
client = TestClient(api.app)
r = client.get(f"/recommendations/{demo}")
pd.Series({"health": client.get("/health").json()["status"], "status_code": r.status_code, "items": len(r.json()["items"]), "hits_in_top12": sum(i["bought_next_week"] for i in r.json()["items"]), "cold_start_strategy": client.get("/recommendations/new-shopper?age_band=25-34").json()["strategy"]})

health                                 ok
status_code                           200
items                                  12
hits_in_top12                           2
cold_start_strategy    cold_start_popular
dtype: object